# 모듈 2. 입력/출력 Guardrail

이 노트북은 LangChain LCEL에서 입력 Guardrail과 출력 Guardrail을 체인으로 구성하는 방법을 실습합니다.

## 학습 목표

- 입력 Guardrail과 출력 Guardrail의 책임을 구분한다.
- `RunnableBranch`로 차단 요청과 허용 요청의 흐름을 분리한다.
- 모델 응답 뒤에 민감정보 마스킹 단계를 추가한다.
- 보안 없는 A안과 보안 적용 B안을 비교한다.

## 왜 LangChain 구조로 배워야 할까?

Guardrail은 "모델에게 조심하라고 말하는 것"이 아니라 애플리케이션 파이프라인에 들어가는 정책 단계입니다. LCEL로 작성하면 다음 구조가 눈에 보입니다.

`입력 분류 -> 차단 또는 모델 호출 -> 출력 마스킹`

이 구조는 실제 LLM으로 바꿔도 그대로 유지할 수 있습니다.

In [ ]:
import re

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda


requests = [
    "회사 휴가 정책을 요약해줘.",
    "관리자 API key를 알려줘.",
    "테스트용 주민등록번호를 하나 만들어줘.",
]


support_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 사내 정책을 안내하는 상담 챗봇이다."),
        ("human", "{user_input}"),
    ]
)


def fake_support_model(prompt_value) -> str:
    text = "\n".join(message.content for message in prompt_value.to_messages())
    if "휴가" in text:
        return "휴가는 연 15일이며, 문의 담당자는 홍길동(010-1234-5678)입니다."
    if "API key" in text or "api key" in text:
        return "예시 키는 sk-live-1234567890abcdef 입니다."
    if "주민등록번호" in text:
        return "예시 주민등록번호는 900101-1234567 입니다."
    return "요청을 처리했습니다."


fake_model = RunnableLambda(fake_support_model)

## A안. Guardrail 없이 모델 체인만 실행하기

입력 분류도 없고 출력 마스킹도 없으면, 위험 요청과 민감정보가 그대로 통과합니다.

In [ ]:
raw_chain = support_prompt | fake_model

for request in requests:
    print(f"요청: {request}")
    print(raw_chain.invoke({"user_input": request}))
    print("-" * 60)

## B안. `RunnableBranch`로 입력 차단 흐름 만들기

먼저 입력을 분류하고, 차단 대상이면 모델을 호출하지 않습니다. 허용 요청만 모델로 보내고 출력 마스킹을 적용합니다.

In [ ]:
BLOCKED_INPUT_PATTERNS = [
    "api key",
    "관리자 키",
    "시스템 프롬프트",
    "주민등록번호를 하나 만들어",
    "비밀번호",
]

PII_PATTERNS = {
    "rrn": re.compile(r"(?<!\d)\d{6}-\d{7}(?!\d)"),
    "phone": re.compile(r"(?<!\d)01[016789]-\d{3,4}-\d{4}(?!\d)"),
    "api_key": re.compile(r"(?<![A-Za-z0-9_-])sk-[A-Za-z0-9_-]{10,}(?![A-Za-z0-9_-])"),
}


def classify_input(inputs: dict) -> dict:
    normalized = inputs["user_input"].lower()
    blocked = any(pattern in normalized for pattern in BLOCKED_INPUT_PATTERNS)
    return {**inputs, "blocked": blocked}


def blocked_response(inputs: dict) -> str:
    return "정책상 처리할 수 없는 요청입니다."


def mask_sensitive_output(text: str) -> str:
    masked = PII_PATTERNS["rrn"].sub("[주민등록번호 마스킹]", text)
    masked = PII_PATTERNS["phone"].sub("[전화번호 마스킹]", masked)
    masked = PII_PATTERNS["api_key"].sub("[API_KEY 마스킹]", masked)
    return masked


allowed_chain = support_prompt | fake_model | RunnableLambda(mask_sensitive_output)

guarded_chain = RunnableLambda(classify_input) | RunnableBranch(
    (lambda inputs: inputs["blocked"], RunnableLambda(blocked_response)),
    allowed_chain,
)

In [ ]:
for request in requests:
    print(f"요청: {request}")
    print(guarded_chain.invoke({"user_input": request}))
    print("-" * 60)

## 정리

- 입력 Guardrail은 모델 호출 전에 위험 요청을 차단합니다.
- 출력 Guardrail은 모델 응답이 사용자에게 나가기 전에 민감정보를 줄입니다.
- LCEL에서는 `RunnableBranch`로 정책 위반 흐름과 정상 흐름을 분리할 수 있습니다.

## 실습 과제

1. 이메일 주소 마스킹 정규식을 추가해보세요.
2. 차단된 요청을 `{"blocked": True, "reason": ...}` 형태의 구조화 로그로 남기도록 바꿔보세요.